# Expanded ensemble testing

13 checkpoints exist from today's runs, but the last notebook only combined 3. This evaluates every checkpoint individually first (quick, no retraining), ranks them, then tests a few different ensemble compositions - not just "bigger is better," since including weak or highly-correlated checkpoints can dilute a strong ensemble rather than help it.

**Important framing for the thesis**: an ensemble result is not a fair like-for-like comparison against Li et al.'s single ST-GCN model - it's a separate, clearly-labeled finding, not a replacement for the single-model comparison. Report both.

### setup

In [1]:
import os
import sys
import glob

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, top_k_accuracy_score

sys.path.insert(0, "/workspace/badminton-honours-project")
import dataset as ds

import mmaction.models  # noqa: F401
from mmaction.utils import register_all_modules
register_all_modules()
from mmaction.registry import MODELS

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda:0


### data

In [2]:
manifest = ds.load_filtered_manifest(
    "/workspace/data/split_manifest.csv",
    "/workspace/data/extraction_log.csv",
)
class_names = sorted(manifest["class_name"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_names)

test_manifest = manifest[manifest["split"] == "test"].reset_index(drop=True)
KEYPOINTS_ROOT = "/workspace/data/keypoints_smoothed"
test_ds = ds.BadmintonPoseDataset(test_manifest, KEYPOINTS_ROOT, class_to_id)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"{num_classes} classes, test {len(test_ds)}")


dropped 4 manually-excluded clips from the manifest (7814 remain)
18 classes, test 782


### locate every checkpoint on disk

In [3]:
search_dirs = [
    "/workspace/checkpoints/stgcn_paper_faithful",
    "/workspace/checkpoints/stgcn_paper_faithful_advanced",
    "/workspace/checkpoints/stgcn_paper_faithful_combined_reg",
    "/workspace/checkpoints/stgcn_combined_reg_cosine",
    "/workspace/checkpoints/stgcn_ablation",
    "/workspace/checkpoints/stgcn_cosine_evidence_based",
    "/workspace/checkpoints/stgcn_cosine_ablation",
]

found_checkpoints = {}
for d in search_dirs:
    for f in glob.glob(os.path.join(d, "*.pt")):
        found_checkpoints[os.path.basename(f)] = f

print(f"{len(found_checkpoints)} checkpoints found")


13 checkpoints found


### build model, load checkpoint, evaluate - same architecture used throughout

In [4]:
def load_model(ckpt_path):
    model_cfg = dict(
        type="RecognizerGCN",
        backbone=dict(
            type="STGCN",
            graph_cfg=dict(layout="coco", mode="stgcn_spatial"),
            in_channels=3, base_channels=64, ch_ratio=2,
            num_stages=9, inflate_stages=[4, 7], down_stages=[4, 7],
        ),
        cls_head=dict(type="GCNHead", num_classes=num_classes, in_channels=256,
                      loss_cls=dict(type="CrossEntropyLoss")),
    )
    model = MODELS.build(model_cfg).to(device)
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def batch_to_stgcn_input(batch):
    kp = batch["keypoints"].float()
    return kp.unsqueeze(1).unsqueeze(1)


@torch.no_grad()
def get_probs(model, loader):
    all_labels, all_probs = [], []
    for batch in loader:
        x = batch_to_stgcn_input(batch).to(device)
        y = batch["label"].to(device)
        logits = model(x, mode="tensor", stage="head")
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_labels.append(y.cpu().numpy())
        all_probs.append(probs)
    return np.concatenate(all_labels), np.concatenate(all_probs)


def metrics_from_probs(labels, probs):
    preds = probs.argmax(axis=1)
    return {
        "top1": accuracy_score(labels, preds),
        "top5": top_k_accuracy_score(labels, probs, k=5, labels=list(range(num_classes))),
        "mca": balanced_accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


### evaluate every checkpoint individually

Also keeps each checkpoint's raw probability array in memory - needed for the ensemble combinations below without reloading and re-running every model again per combination.

In [5]:
individual_results = {}
individual_probs = {}
test_labels = None

for name, path in found_checkpoints.items():
    model = load_model(path)
    labels, probs = get_probs(model, test_loader)
    test_labels = labels
    individual_probs[name] = probs
    individual_results[name] = metrics_from_probs(labels, probs)
    r = individual_results[name]
    print(f"{name:45s} top1={r['top1']:.4f}  mca={r['mca']:.4f}  macro_f1={r['macro_f1']:.4f}")
    del model
    torch.cuda.empty_cache()


stgcn_paper_faithful_basic_best.pt            top1=0.6905  mca=0.5925  macro_f1=0.6052
stgcn_paper_faithful_advanced_best.pt         top1=0.7072  mca=0.6155  macro_f1=0.6309
stgcn_combined_reg_best.pt                    top1=0.6969  mca=0.6089  macro_f1=0.6097
stgcn_combined_reg_cosine_best.pt             top1=0.7238  mca=0.5989  macro_f1=0.6192
weight_decay_only_best.pt                     top1=0.7148  mca=0.6304  macro_f1=0.6413
tcn_dropout_only_best.pt                      top1=0.6880  mca=0.5933  macro_f1=0.6003
label_smoothing_only_best.pt                  top1=0.7174  mca=0.6042  macro_f1=0.6244
baseline_repeat_best.pt                       top1=0.7020  mca=0.6017  macro_f1=0.6233
stgcn_cosine_evidence_based_best.pt           top1=0.7148  mca=0.5690  macro_f1=0.5931
cosine_tcn_dropout_best.pt                    top1=0.6893  mca=0.6000  macro_f1=0.5920
cosine_label_smoothing_best.pt                top1=0.6995  mca=0.5781  macro_f1=0.5915
cosine_weight_decay_best.pt                

### ranked table - individual checkpoint performance

In [6]:
individual_df = pd.DataFrame(individual_results).T.sort_values("top1", ascending=False)
individual_df


,top1,top5,mca,macro_f1
stgcn_combined_reg_cosine_best.pt,0.723785,0.924552,0.598933,0.619223
label_smoothing_only_best.pt,0.717391,0.938619,0.604215,0.624397
weight_decay_only_best.pt,0.714834,0.943734,0.630427,0.641292
stgcn_cosine_evidence_based_best.pt,0.714834,0.924552,0.569037,0.593083
stgcn_paper_faithful_advanced_best.pt,0.707161,0.933504,0.615522,0.630857
baseline_repeat_best.pt,0.702046,0.947570,0.601660,0.623346
cosine_label_smoothing_best.pt,0.699488,0.915601,0.578110,0.591539
cosine_none_best.pt,0.696931,0.945013,0.566581,0.584309
stgcn_combined_reg_best.pt,0.696931,0.941176,0.608900,0.609733
stgcn_paper_faithful_basic_best.pt,0.690537,0.947570,0.592477,0.605246


### test a few different ensemble compositions

Not just "all 13" - a large ensemble including weak or near-duplicate checkpoints (e.g. several cosine runs that all converged similarly) can dilute a strong result rather than improve it. Testing a spread of composition sizes/strategies to see what actually works best, rather than assuming bigger is automatically better.

In [7]:
def evaluate_ensemble(names):
    probs_sum = sum(individual_probs[n] for n in names)
    probs_avg = probs_sum / len(names)
    return metrics_from_probs(test_labels, probs_avg)


ranked_names = individual_df.index.tolist()

ensemble_candidates = {
    "top_3":  ranked_names[:3],
    "top_5":  ranked_names[:5],
    "top_7":  ranked_names[:7],
    "all_13": ranked_names,
}

ensemble_results = {}
for name, members in ensemble_candidates.items():
    ensemble_results[name] = evaluate_ensemble(members)
    r = ensemble_results[name]
    print(f"{name:10s} ({len(members)} models)  top1={r['top1']:.4f}  "
          f"top5={r['top5']:.4f}  mca={r['mca']:.4f}  macro_f1={r['macro_f1']:.4f}")


top_3      (3 models)  top1=0.7379  top5=0.9399  mca=0.6277  macro_f1=0.6501
top_5      (5 models)  top1=0.7366  top5=0.9476  mca=0.6201  macro_f1=0.6452
top_7      (7 models)  top1=0.7315  top5=0.9501  mca=0.6213  macro_f1=0.6441
all_13     (13 models)  top1=0.7366  top5=0.9501  mca=0.6314  macro_f1=0.6503


### final comparison - best single model, best ensemble, and the benchmark

In [8]:
LI_ET_AL_BENCHMARK = {"top1": 0.7441, "top5": 0.9376, "mca": 0.6144, "macro_f1": None}

best_single_name = individual_df.index[0]
best_ensemble_name = max(ensemble_results, key=lambda k: ensemble_results[k]["top1"])

summary = pd.DataFrame([
    {"result": f"best single model ({best_single_name})", **individual_results[best_single_name]},
    {"result": f"best ensemble ({best_ensemble_name}, {len(ensemble_candidates[best_ensemble_name])} models)", **ensemble_results[best_ensemble_name]},
    {"result": "Li et al. benchmark (single model)", **LI_ET_AL_BENCHMARK},
]).set_index("result")

summary


,top1,top5,mca,macro_f1
result,,,,
best single model (stgcn_combined_reg_cosine_best.pt),0.723785,0.924552,0.598933,0.619223
"best ensemble (top_3, 3 models)",0.737852,0.939898,0.627710,0.650073
Li et al. benchmark (single model),0.744100,0.937600,0.614400,NaN


---

Two separate, honest numbers for the thesis: **best single model** is the fair, apples-to-apples comparison against Li et al.'s single ST-GCN - report this as the primary result. **Best ensemble** is a real, additional finding worth reporting too, but clearly labeled as an ensemble (different inference cost, not a substitute for the single-model comparison) rather than presented as if it were the same kind of result as the benchmark.